# 00 — Explore EB-NeRD (Demo Bundle)

Exploratory Data Analysis for the **EB-NeRD demo** dataset (Danish, Parquet format).

This notebook inventories file structure, row counts, schema, timestamp ranges, click-through rate, per-user / per-article distributions, missing text, embeddings, and confirms Danish-language content requiring Unicode-aware tokenization.

In [1]:
import polars as pl
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path(".").resolve().parent
EBNERD_DIR = PROJECT_ROOT / "data" / "raw" / "ebnerd"

print(f"Project root: {PROJECT_ROOT}")
print(f"EB-NeRD dir: {EBNERD_DIR}")

Project root: /home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir
EB-NeRD dir: /home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/ebnerd


## 1. File Listing and Sizes

In [2]:
import subprocess
result = subprocess.run(["ls", "-lhR", str(EBNERD_DIR)], capture_output=True, text=True)
print(result.stdout)

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/ebnerd:
total 36M
-rw-rw-r-- 1 shrawani shrawani  15M Aug 14 18:31 articles.parquet
-rw-rw-r-- 1 shrawani shrawani  21M Aug 14 18:31 ebnerd_demo.zip
drwxrwxr-x 4 shrawani shrawani 4.0K Aug 14 18:31 __MACOSX
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:31 train
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:31 validation

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/ebnerd/__MACOSX:
total 8.0K
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:31 train
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:31 validation

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/ebnerd/__MACOSX/train:
total 0

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/ebnerd/__MACOSX/validation:
total 0

/home/shrawani/Desktop/sem5/Information Retrieva

## 2. Row Counts (Lazy Polars Scan)

In [3]:
parquet_files = [
    ("articles.parquet", EBNERD_DIR / "articles.parquet"),
    ("train/behaviors.parquet", EBNERD_DIR / "train" / "behaviors.parquet"),
    ("train/history.parquet", EBNERD_DIR / "train" / "history.parquet"),
    ("validation/behaviors.parquet", EBNERD_DIR / "validation" / "behaviors.parquet"),
    ("validation/history.parquet", EBNERD_DIR / "validation" / "history.parquet"),
]

for name, path in parquet_files:
    count = pl.scan_parquet(path).select(pl.count()).collect().item()
    size_mb = path.stat().st_size / 1024**2
    print(f"  {name:40s} {count:>8,} rows   ({size_mb:.2f} MB)")

  articles.parquet                           11,777 rows   (14.57 MB)
  train/behaviors.parquet                    24,724 rows   (1.08 MB)
  train/history.parquet                       1,590 rows   (2.44 MB)
  validation/behaviors.parquet               25,356 rows   (1.15 MB)
  validation/history.parquet                  1,562 rows   (2.27 MB)


/tmp/ipykernel_39202/2706453999.py:10: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  count = pl.scan_parquet(path).select(pl.count()).collect().item()


## 3. Schema per File

In [4]:
for name, path in parquet_files:
    df = pl.read_parquet(path, n_rows=0)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    for c, d in zip(df.columns, df.dtypes):
        print(f"  {c:40s} {str(d)}")


  articles.parquet
  article_id                               Int32
  title                                    String
  subtitle                                 String
  last_modified_time                       Datetime(time_unit='us', time_zone=None)
  premium                                  Boolean
  body                                     String
  published_time                           Datetime(time_unit='us', time_zone=None)
  image_ids                                List(Int64)
  article_type                             String
  url                                      String
  ner_clusters                             List(String)
  entity_groups                            List(String)
  topics                                   List(String)
  category                                 Int16
  subcategory                              List(Int16)
  category_str                             String
  total_inviews                            Int32
  total_pageviews                   

## 4. Sample Rows (head 5)

In [5]:
for name, path in parquet_files:
    print(f"\n--- {name} ---")
    df = pl.read_parquet(path, n_rows=5)
    # Print with narrower width for readability
    with pl.Config(tbl_cols=8, tbl_width_chars=120, fmt_str_lengths=40):
        print(df)


--- articles.parquet ---
shape: (5, 21)
┌────────────┬──────────────┬──────────────┬──────────────┬───┬──────────────┬─────────────┬─────────────┬─────────────┐
│ article_id ┆ title        ┆ subtitle     ┆ last_modifie ┆ … ┆ total_pagevi ┆ total_read_ ┆ sentiment_s ┆ sentiment_l │
│ ---        ┆ ---          ┆ ---          ┆ d_time       ┆   ┆ ews          ┆ time        ┆ core        ┆ abel        │
│ i32        ┆ str          ┆ str          ┆ ---          ┆   ┆ ---          ┆ ---         ┆ ---         ┆ ---         │
│            ┆              ┆              ┆ datetime[μs] ┆   ┆ i32          ┆ f32         ┆ f32         ┆ str         │
╞════════════╪══════════════╪══════════════╪══════════════╪═══╪══════════════╪═════════════╪═════════════╪═════════════╡
│ 3037230    ┆ Ishockey-spi ┆ ISHOCKEY:    ┆ 2023-06-29   ┆ … ┆ null         ┆ null        ┆ 0.9752      ┆ Negative    │
│            ┆ ller: Jeg    ┆ Ishockey-spi ┆ 06:20:57     ┆   ┆              ┆             ┆             ┆      

## 5. Timestamp Range (Impressions)

In [6]:
for split in ["train", "validation"]:
    beh = pl.read_parquet(EBNERD_DIR / split / "behaviors.parquet")
    ts_min = beh["impression_time"].min()
    ts_max = beh["impression_time"].max()
    print(f"{split}: {ts_min} → {ts_max}")

train: 2023-05-18 07:00:03 → 2023-05-25 06:59:52
validation: 2023-05-25 07:00:15 → 2023-06-01 06:59:33


## 6. Click-Through Rate

In [7]:
for split in ["train", "validation"]:
    beh = pl.read_parquet(EBNERD_DIR / split / "behaviors.parquet")
    
    total_pairs = beh["article_ids_inview"].list.len().sum()
    positive_clicks = beh["article_ids_clicked"].list.len().sum()
    ctr = positive_clicks / total_pairs if total_pairs else 0
    
    print(f"{split}: {positive_clicks:,} clicks / {total_pairs:,} pairs = CTR {ctr:.4f} ({ctr*100:.2f}%)")

train: 24,888 clicks / 278,139 pairs = CTR 0.0895 (8.95%)
validation: 25,505 clicks / 304,915 pairs = CTR 0.0836 (8.36%)


## 7. Clicks per User Distribution

In [8]:
for split in ["train", "validation"]:
    beh = pl.read_parquet(EBNERD_DIR / split / "behaviors.parquet")
    
    user_clicks = (
        beh.select(
            pl.col("user_id"),
            pl.col("article_ids_clicked").list.len().alias("n_clicks")
        )
        .group_by("user_id")
        .agg(pl.col("n_clicks").sum())
        .sort("n_clicks")
    )
    
    vals = user_clicks["n_clicks"].to_list()
    n = len(vals)
    print(f"\n{split} — {n:,} users with ≥1 click")
    print(f"  min={vals[0]}, max={vals[-1]}, mean={sum(vals)/n:.2f}")
    print(f"  median={vals[n//2]}, p75={vals[int(n*0.75)]}, p90={vals[int(n*0.90)]}")
    print(f"  p95={vals[int(n*0.95)]}, p99={vals[int(n*0.99)]}")


train — 1,590 users with ≥1 click
  min=1, max=177, mean=15.65
  median=9, p75=22, p90=39
  p95=50, p99=77

validation — 1,562 users with ≥1 click
  min=1, max=122, mean=16.33
  median=10, p75=24, p90=40
  p95=54, p99=78


## 8. Impressions per Article Distribution

In [9]:
for split in ["train", "validation"]:
    beh = pl.read_parquet(EBNERD_DIR / split / "behaviors.parquet")
    
    # Explode inview articles
    article_counts = (
        beh.select(pl.col("article_ids_inview").explode().alias("article_id"))
        .group_by("article_id")
        .agg(pl.count().alias("n_impressions"))
        .sort("n_impressions", descending=True)
    )
    
    vals = article_counts["n_impressions"].to_list()
    n = len(vals)
    print(f"\n{split} — {n:,} unique articles in impressions")
    print(f"  min={vals[-1]}, max={vals[0]}, mean={sum(vals)/n:.2f}, median={vals[n//2]}")
    print(f"  Top 10: {vals[:10]}")


train — 2,478 unique articles in impressions
  min=1, max=3649, mean=112.24, median=15
  Top 10: [3649, 2031, 1944, 1672, 1639, 1305, 1185, 1165, 1144, 1051]

validation — 2,738 unique articles in impressions
  min=1, max=2323, mean=111.36, median=12
  Top 10: [2323, 1904, 1683, 1605, 1600, 1528, 1066, 1052, 996, 978]


/tmp/ipykernel_39202/2068437289.py:6: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  beh.select(pl.col("article_ids_inview").explode().alias("article_id"))
/tmp/ipykernel_39202/2068437289.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("n_impressions"))


## 9. Missing Abstract/Body/Subtitle

In [10]:
articles = pl.read_parquet(EBNERD_DIR / "articles.parquet")
n = articles.shape[0]
print(f"Total articles: {n:,}")

for col in ["title", "subtitle", "body"]:
    null_count = articles[col].null_count()
    empty_count = articles.filter(pl.col(col) == "").shape[0]
    total_missing = null_count + empty_count
    print(f"  {col}: null={null_count}, empty_str={empty_count}, "
          f"total_missing={total_missing} ({total_missing/n*100:.1f}%)")

Total articles: 11,777
  title: null=0, empty_str=0, total_missing=0 (0.0%)
  subtitle: null=0, empty_str=803, total_missing=803 (6.8%)
  body: null=0, empty_str=933, total_missing=933 (7.9%)


## 10. Article Embeddings (EB-NeRD specific)

The full EB-NeRD dataset provides pre-computed article embeddings (Word2Vec + multilingual BERT) in separate zip files. The **demo bundle does not include embedding files**.

In [11]:
articles = pl.read_parquet(EBNERD_DIR / "articles.parquet")

# Check for any embedding-like columns
emb_cols = [c for c in articles.columns if "emb" in c.lower() or "vector" in c.lower()]
list_numeric_cols = [c for c in articles.columns 
                     if str(articles[c].dtype).startswith("List(Float") 
                     or str(articles[c].dtype).startswith("List(Int")]

print(f"Embedding columns: {emb_cols if emb_cols else 'None'}")
print(f"List-numeric columns (potential embeddings): {list_numeric_cols if list_numeric_cols else 'None'}")
print()
print("⚠ Demo bundle does NOT include embedding files.")
print("  Full dataset provides: article_embeddings.parquet with Word2Vec + multilingual BERT vectors")
print("  Download ebnerd_small.zip or the embeddings zips separately when ready.")

Embedding columns: None
List-numeric columns (potential embeddings): ['image_ids', 'subcategory']

⚠ Demo bundle does NOT include embedding files.
  Full dataset provides: article_embeddings.parquet with Word2Vec + multilingual BERT vectors
  Download ebnerd_small.zip or the embeddings zips separately when ready.


## 11. Danish Language Confirmation

EB-NeRD text is in Danish. The presence of æ/ø/å characters confirms that **Unicode-aware tokenization** is required — English stopword lists and ASCII-based tokenizers will not work correctly.

In [12]:
articles = pl.read_parquet(EBNERD_DIR / "articles.parquet")

# Sample titles
print("Sample Danish titles:")
for title in articles["title"].head(10).to_list():
    print(f"  • {title}")

# Check for Danish-specific characters
danish_chars = set()
for title in articles["title"].to_list():
    for ch in title:
        if ch in "æøåÆØÅ":
            danish_chars.add(ch)

print(f"\nDanish characters found: {sorted(danish_chars)}")
print("\n⚠ Note: Do NOT use English tokenizers/stopword lists.")
print("  æ, ø, å are distinct characters, not diacritical variants.")
print("  Use a Unicode-aware tokenizer (e.g., spaCy 'da' model or custom regex).")

Sample Danish titles:
  • Ishockey-spiller: Jeg troede jeg skulle dø
  • Prins Harry tvunget til dna-test
  • Rådden kørsel på blå plader
  • Mærsk-arvinger i livsfare
  • Skød svigersøn gennem babydyne
  • Zoo-tårnet 100 år
  • Tævet ihjel på tre kvarter
  • Denne kæp kan fælde voldtægtsmand
  • Morder truer med nyt drab
  • Pædofil må stadig undervise børn

Danish characters found: ['Å', 'Æ', 'Ø', 'å', 'æ', 'ø']

⚠ Note: Do NOT use English tokenizers/stopword lists.
  æ, ø, å are distinct characters, not diacritical variants.
  Use a Unicode-aware tokenizer (e.g., spaCy 'da' model or custom regex).
